# Ficha automática de mercado y entorno — Venture One SAS

**Reto 13 · EstudIA · Análisis de datos · Capa 2 · Nivel básico**

Objetivo: generar, para cada inmueble de prueba, una ficha JSON con:
1. **Mercado:** precio/m² de comparables en el mismo estado y una regla de confianza.
2. **Entorno:** estación de transporte más cercana y equipamientos dentro del radio de búsqueda.
3. **Norma:** tratamiento del POT (cuando la capa esté disponible).
4. **Calidad:** duplicados, datos inválidos y datos faltantes.

Lea antes de empezar:
- `docs/diccionario_datos.md` — columnas y calidad de cada archivo.
- `config/parametros.yaml` — radio, estados y regla de confianza. **Ningún número va fijo en el código.**
- `docs/plantilla_ficha.md` y `ejemplos/ficha_ejemplo.json` — formato de salida.

Reglas del reto: use solo los datos entregados. No haga scraping de portales. No use datos personales.

In [ ]:
# 1. Preparar el entorno: clona el repositorio en Colab y se ubica en su raíz.
import os, subprocess, sys

REPO_URL = "https://github.com/solhecoo/ventureone-reto-ficha.git"
CARPETA = "ventureone-reto-ficha"

if "google.colab" in sys.modules and not os.path.exists(CARPETA):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
if os.path.exists(CARPETA):
    os.chdir(CARPETA)
elif os.path.exists("../config/parametros.yaml"):
    os.chdir("..")  # ejecución local desde notebooks/
assert os.path.exists("config/parametros.yaml"), "No se encontró la raíz del repositorio"
print("Raíz del repositorio:", os.getcwd())

In [ ]:
# 2. Cargar parámetros y datos.
import csv, json
import yaml

PARAMS = yaml.safe_load(open("config/parametros.yaml", encoding="utf-8"))
ANUNCIOS = list(csv.DictReader(open("data/anuncios_muestra.csv", encoding="utf-8")))
INMUEBLES = list(csv.DictReader(open("data/inmuebles_prueba.csv", encoding="utf-8")))
TRANSPORTE = json.load(open("data/geo/transporte.geojson", encoding="utf-8"))
EQUIPAMIENTOS = json.load(open("data/geo/equipamientos.geojson", encoding="utf-8"))
POT = json.load(open(PARAMS["norma"]["archivo_pot"], encoding="utf-8")) if os.path.exists(PARAMS["norma"]["archivo_pot"]) else None

print(f"Anuncios: {len(ANUNCIOS)} · Inmuebles de prueba: {len(INMUEBLES)}")
print(f"Estaciones: {len(TRANSPORTE['features'])} · Equipamientos: {len(EQUIPAMIENTOS['features'])} · OSM {TRANSPORTE['metadata']['fecha_osm']}")
print("Capa POT:", "cargada" if POT else "pendiente")

## Exploración

Revise la calidad antes de calcular: ¿cuántos `id_anuncio` se repiten? ¿Qué variantes tiene `estado`?
¿Qué columnas están vacías? Registre lo que encuentre: la ficha debe declarar los datos faltantes.

In [ ]:
import pandas as pd

df = pd.read_csv("data/anuncios_muestra.csv")
display(df.head())
print("Filas por zona:\n", df["zona"].value_counts(), "\n")
print("Variantes de estado:\n", df["estado"].value_counts(dropna=False), "\n")
print("id_anuncio repetidos:", df["id_anuncio"].dropna().duplicated().sum())
print("Vacíos por columna:\n", df.isna().sum())

## Su solución

Complete las funciones. Cada una recibe `PARAMS` para leer las reglas.
La función `haversine_m` se entrega lista.

In [ ]:
import math

def haversine_m(lat1, lon1, lat2, lon2):
    """Distancia en metros entre dos puntos en grados decimales."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6_371_008.8 * math.asin(math.sqrt(a))


def limpiar_anuncios(anuncios, params):
    """Normaliza estados, elimina duplicados y marca los anuncios inválidos."""
    raise NotImplementedError("Paso 1: limpieza")


def bloque_mercado(inmueble, anuncios_limpios, params):
    """Comparables, estadísticas de precio/m² y nivel de confianza."""
    raise NotImplementedError("Paso 2: mercado")


def bloque_entorno(inmueble, transporte, equipamientos, params):
    """Estación más cercana y conteos dentro del radio."""
    raise NotImplementedError("Paso 3: entorno")


def generar_ficha(inmueble):
    """Arma la ficha completa con el formato de ejemplos/ficha_ejemplo.json."""
    raise NotImplementedError("Paso 4: ficha")

In [ ]:
# Generar una ficha por inmueble en salidas/participante/
os.makedirs("salidas/participante", exist_ok=True)
for inmueble in INMUEBLES:
    ficha = generar_ficha(inmueble)
    with open(f"salidas/participante/{inmueble['id_inmueble']}.json", "w", encoding="utf-8") as f:
        json.dump(ficha, f, ensure_ascii=False, indent=2)
print("Fichas generadas:", sorted(os.listdir("salidas/participante")))

## Validación contra el cálculo manual

`validacion/calculo_manual.md` calcula a mano INM-04, INM-06 e INM-09.
La celda compara sus fichas con esos valores y lista cada diferencia.

In [ ]:
resultado = subprocess.run([sys.executable, "validacion/validar.py", "salidas/participante"], capture_output=True, text=True)
print(resultado.stdout or resultado.stderr)
print("✅ Coincide con el cálculo manual" if resultado.returncode == 0 else "❌ Revise las diferencias")